# AegisGuard 1.3.0 LMS Audio briefing

Two hosts (**Mira** and **Rex**) talk through every shipped change from public GitHub Release **1.2.7** through current `V1.3.0` HEAD, including Claim Status.

**How to listen**

1. Open **LM Studio**, load a TTS model (Kokoro is a good default), and start the local server (`http://127.0.0.1:1234`).
2. Run all cells below. Each turn plays in this notebook; the full show is also written to `out/aegisguard-1.3.0-briefing.wav`.
3. Optional CLI: `python generate_briefing.py` in this folder.

This is **not** a public GitHub Release. Latest published release stays **1.2.7**.

In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd()
if not (HERE / "generate_briefing.py").exists():
    HERE = Path(r"C:\Users\Kai\Dropbox\AegisGuard 1.3.0 - repo\lms-audio")
sys.path.insert(0, str(HERE))

import generate_briefing as briefing

LMS_BASE = "http://127.0.0.1:1234"
VOICE_MIRA = "af_bella"   # Kokoro female
VOICE_REX = "am_michael"  # Kokoro male
TTS_MODEL = None          # auto-pick Kokoro / Orpheus if listed

probe = briefing.probe_lms(LMS_BASE)
probe

## Script preview

Full copy lives in `aegisguard_1_3_0_briefing_script.md`. Numbers and product names below match git history and contract tests: TradeStalls, ClaimBlocks, Guest Passes, Lockdown, Arena, schema 1286.

In [ ]:
for i, (speaker, line) in enumerate(briefing.DIALOGUE, start=1):
    name = "Mira" if speaker == "mira" else "Rex"
    print(f"{i:02d} {name}: {line}\n")

## Speak the briefing

Run this cell once LM Studio TTS is loaded. It synthesizes each turn, plays it, then concatenates `out/aegisguard-1.3.0-briefing.wav`.

In [ ]:
from IPython.display import Audio, display

if not probe.get("ok"):
    raise RuntimeError(
        probe.get("error")
        or "LM Studio is not reachable. Start the app, load TTS, enable the local server on port 1234."
    )

tts_model = briefing.pick_tts_model(probe.get("models") or [], TTS_MODEL)
print("Using TTS model:", tts_model)

blobs = []
for i, (speaker, line) in enumerate(briefing.DIALOGUE, start=1):
    voice = VOICE_MIRA if speaker == "mira" else VOICE_REX
    name = "Mira" if speaker == "mira" else "Rex"
    print(f"[{i}/{len(briefing.DIALOGUE)}] {name}")
    audio = briefing.synthesize(LMS_BASE, tts_model, voice, line)
    blobs.append(audio)
    display(Audio(audio, autoplay=(i == 1)))

out = briefing.concat_wavs(blobs, HERE / "out" / "aegisguard-1.3.0-briefing.wav")
print("Full briefing:", out)
display(Audio(filename=str(out)))